## 1. Environment Setup & Data Loading

This section configures the Jupyter environment and loads our preprocessed catalog data. 

**Key Configuration:**
* **`%autoreload 2`**: Automatically reloads imported modules. This allows us to actively develop `src/reco_nova/content_model.py` in an external editor and instantly test the changes here without restarting the kernel.
* **Path Management**: Dynamically routes `sys.path` to the project root so Python can discover our custom `reco_nova` package, regardless of whether this notebook is executed from the root directory or the `notebooks/` subfolder.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from importlib import reload

# Add the src/ directory to Python's path so it can find reco_nova
current_dir = Path.cwd()
src_path = current_dir.parent / "src" if current_dir.name == "notebooks" else current_dir / "src"
sys.path.insert(0, str(src_path))

import reco_nova.content_model as content_model

# Reload the module so notebook sessions pick up the latest code changes
content_model = reload(content_model)
ContentRecommender = content_model.ContentRecommender

# Build the exact path to the parquet file and load it
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
items_path = project_root / "data" / "processed" / "items_clean.parquet"

print(f"Loading data from: {items_path}")
items = pd.read_parquet(items_path)
display(items.head(5))

/Users/chikire/miniforge3/envs/reco-nova/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data from: /Users/chikire/mds/Reco-Nova/data/processed/items_clean.parquet


,article_id,prod_name,product_type_name,product_group_name,department_name,section_name,garment_group_name,detail_desc,image_path,has_image,item_text,item_idx
0,108775015,strap top,vest top,garment upper body,jersey basic,womens everyday basics,jersey basic,jersey top with narrow shoulder straps.,data/raw/images/010/0108775015.jpg,True,strap top vest top garment upper body jersey b...,0
1,108775044,strap top,vest top,garment upper body,jersey basic,womens everyday basics,jersey basic,jersey top with narrow shoulder straps.,data/raw/images/010/0108775044.jpg,True,strap top vest top garment upper body jersey b...,1
2,108775051,strap top (1),vest top,garment upper body,jersey basic,womens everyday basics,jersey basic,jersey top with narrow shoulder straps.,data/raw/images/010/0108775051.jpg,True,strap top (1) vest top garment upper body jers...,<NA>
3,110065001,op t-shirt (idro),bra,underwear,clean lingerie,womens lingerie,"under-, nightwear","microfibre t-shirt bra with underwired, moulde...",data/raw/images/011/0110065001.jpg,True,op t-shirt (idro) bra underwear clean lingerie...,2
4,110065002,op t-shirt (idro),bra,underwear,clean lingerie,womens lingerie,"under-, nightwear","microfibre t-shirt bra with underwired, moulde...",data/raw/images/011/0110065002.jpg,True,op t-shirt (idro) bra underwear clean lingerie...,3


## 2. Engine Initialization & Feature Building

Here we initialize the `ContentRecommender` and build our two Item-to-Item (I2I) similarity engines. 
* **TF-IDF**: Builds a sparse matrix mapping exact keyword frequencies (capped at 5,000 features). Best for matching specific attributes (e.g., "cotton", "v-neck").
* **Embeddings (FAISS)**: Uses `all-MiniLM-L6-v2` to generate dense semantic vectors for each item, stored in a FAISS (Facebook AI Similarity Search) index for lightning-fast nearest-neighbor retrieval. Best for capturing the overall "vibe" or style of an item, even if exact keywords differ.

*Note: This cell handles the computational heavy lifting. Run it once, and the indices will remain in memory for rapid querying.*

In [2]:
# Initialize the Recommender
recommender = ContentRecommender(items)

# Build the features
recommender.build_tfidf()
recommender.build_embeddings()

Building TF-IDF Matrix...
TF-IDF Matrix Shape: (105542, 5000)
Loading SentenceTransformer: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7153.25it/s]


Encoding item texts (this may take a few minutes)...


Batches: 100%|██████████| 413/413 [02:11<00:00,  3.15it/s]


Building FAISS Index for fast similarity search...
FAISS Index successfully built.


## 3. Presentation & Evaluation Helpers

To maintain a clean, production-grade architecture, the core recommendation math and string-matching logic live in `content_model.py`. 

This section contains notebook-specific wrapper functions designed purely for **presentation**. These functions consume the raw data returned by our recommender engine and format it into clean, readable Pandas DataFrames so we can easily evaluate the quality of the recommendations.

In [3]:
from IPython.display import display

def build_query_tables(query_ids, top_n=5, method="tfidf"):
    """Build one recommendation table per query article_id."""
    query_tables = {}

    for query_id in query_ids:
        if method == "tfidf":
            frame = recommender.get_similar_tfidf_df(query_id, top_n=top_n)
        elif method == "embeddings":
            frame = recommender.get_similar_embeddings_df(query_id, top_n=top_n)
        else:
            raise ValueError("method must be 'tfidf' or 'embeddings'")

        if frame.empty:
            continue

        frame = frame.copy()
        frame["method"] = method
        query_tables[query_id] = frame.reset_index(drop=True)

    return query_tables

def build_query_tables_from_names(name_queries, top_n=5, method="tfidf"):
    """Name -> best matching article_id -> recommendation table."""
    resolved_rows = []
    per_query_tables = {}

    for name_query in name_queries:
        resolution = recommender.resolve_name_to_article_id(name_query)
        if resolution is None:
            resolved_rows.append(
                {
                    "query_name": name_query,
                    "article_id": None,
                    "prod_name": None,
                    "match_type": "no_match",
                    "name_match_score": 0.0,
                }
            )
            continue

        resolved_rows.append(resolution)
        rec_map = build_query_tables(
            [resolution["article_id"]],
            top_n=top_n,
            method=method,
        )
        table = rec_map.get(resolution["article_id"], pd.DataFrame()).copy()
        if not table.empty:
            table.insert(0, "query_name", name_query)
            table.insert(1, "resolved_prod_name", resolution["prod_name"])
            table.insert(2, "match_type", resolution["match_type"])
            table.insert(3, "name_match_score", resolution["name_match_score"])
        per_query_tables[name_query] = table

    return pd.DataFrame(resolved_rows), per_query_tables

## 4. Baseline Testing: Canonical ID Queries

Our recommender is fundamentally an **Item-to-Item (I2I)** engine. Under the hood, it strictly uses the canonical `article_id` to perform similarity lookups. 

This cell tests the raw pipeline by passing in exact `article_id`s from our catalog and retrieving their nearest mathematical neighbors.

In [4]:
# Build a small query set from the catalog
query_set = items[["article_id", "item_text"]].head(3).copy()
query_set = query_set.rename(
    columns={
        "article_id": "query_article_id",
        "item_text": "query_item_text",
    }
)

# Run ID-based flow
query_ids = query_set["query_article_id"].tolist()
tfidf_query_tables = build_query_tables(query_ids, top_n=5, method="tfidf")

print("ID-Based Recommendations Generated.\n")

# Loop through the dictionary and display each table
for query_id, table in tfidf_query_tables.items():
    print(f"Query ID: {query_id}")
    display(table)

ID-Based Recommendations Generated.

Query ID: 108775015


,query_article_id,query_item_text,rank,score,article_id,item_text,method
0,108775015,strap top vest top garment upper body jersey b...,1,1.000000,108775051,strap top (1) vest top garment upper body jers...,tfidf
1,108775015,strap top vest top garment upper body jersey b...,2,1.000000,108775044,strap top vest top garment upper body jersey b...,tfidf
2,108775015,strap top vest top garment upper body jersey b...,3,0.877615,538699001,v-neck strap top vest top garment upper body j...,tfidf
3,108775015,strap top vest top garment upper body jersey b...,4,0.877615,538699002,v-neck strap top vest top garment upper body j...,tfidf
4,108775015,strap top vest top garment upper body jersey b...,5,0.877615,538699007,v-neck strap top vest top garment upper body j...,tfidf


Query ID: 108775044


,query_article_id,query_item_text,rank,score,article_id,item_text,method
0,108775044,strap top vest top garment upper body jersey b...,1,1.000000,108775015,strap top vest top garment upper body jersey b...,tfidf
1,108775044,strap top vest top garment upper body jersey b...,2,1.000000,108775051,strap top (1) vest top garment upper body jers...,tfidf
2,108775044,strap top vest top garment upper body jersey b...,3,0.877615,538699001,v-neck strap top vest top garment upper body j...,tfidf
3,108775044,strap top vest top garment upper body jersey b...,4,0.877615,538699002,v-neck strap top vest top garment upper body j...,tfidf
4,108775044,strap top vest top garment upper body jersey b...,5,0.877615,538699007,v-neck strap top vest top garment upper body j...,tfidf


Query ID: 108775051


,query_article_id,query_item_text,rank,score,article_id,item_text,method
0,108775051,strap top (1) vest top garment upper body jers...,1,1.000000,108775015,strap top vest top garment upper body jersey b...,tfidf
1,108775051,strap top (1) vest top garment upper body jers...,2,1.000000,108775044,strap top vest top garment upper body jersey b...,tfidf
2,108775051,strap top (1) vest top garment upper body jers...,3,0.877615,538699001,v-neck strap top vest top garment upper body j...,tfidf
3,108775051,strap top (1) vest top garment upper body jers...,4,0.877615,538699002,v-neck strap top vest top garment upper body j...,tfidf
4,108775051,strap top (1) vest top garment upper body jers...,5,0.877615,538699007,v-neck strap top vest top garment upper body j...,tfidf


## 5. User-Centric Testing: Natural Language Queries

Users don't think in 9-digit `article_id` strings. This cell tests our UI/UX bridge layer. 

The process flows in two steps:
1. **Resolution**: Takes a human-readable text query (e.g., "floral dress") and uses exact, contains, or fuzzy string matching to map it to the best canonical `article_id` in the catalog.
2. **Inference**: Passes that resolved `article_id` into the I2I engine to generate recommendations. 

This simulates how a user might interact with a search bar on a front-end application.

In [5]:
# Run new name-based flow
name_queries = ["strap top", "op t-shirt", "v-neck strap top"]
name_resolution_df, tfidf_name_query_tables = build_query_tables_from_names(
    name_queries,
    top_n=5,
    method="tfidf",
)

print("Name -> article_id resolution")
display(name_resolution_df)

print("\nPer-query recommendation tables (name-based input)")
for query_name, table in tfidf_name_query_tables.items():
    print(f"Query name: {query_name}")
    if table.empty:
        print("No recommendations generated for this query.")
        continue
    display(table)

Name -> article_id resolution


,query_name,article_id,prod_name,match_type,name_match_score
0,strap top,108775015,strap top,exact,1.000000
1,op t-shirt,762928001,boop t-shirt,contains,0.909091
2,v-neck strap top,538699001,v-neck strap top,exact,1.000000



Per-query recommendation tables (name-based input)
Query name: strap top


,query_name,resolved_prod_name,match_type,name_match_score,query_article_id,query_item_text,rank,score,article_id,item_text,method
0,strap top,strap top,exact,1.0,108775015,strap top vest top garment upper body jersey b...,1,1.000000,108775051,strap top (1) vest top garment upper body jers...,tfidf
1,strap top,strap top,exact,1.0,108775015,strap top vest top garment upper body jersey b...,2,1.000000,108775044,strap top vest top garment upper body jersey b...,tfidf
2,strap top,strap top,exact,1.0,108775015,strap top vest top garment upper body jersey b...,3,0.877615,538699001,v-neck strap top vest top garment upper body j...,tfidf
3,strap top,strap top,exact,1.0,108775015,strap top vest top garment upper body jersey b...,4,0.877615,538699002,v-neck strap top vest top garment upper body j...,tfidf
4,strap top,strap top,exact,1.0,108775015,strap top vest top garment upper body jersey b...,5,0.877615,538699007,v-neck strap top vest top garment upper body j...,tfidf


Query name: op t-shirt


,query_name,resolved_prod_name,match_type,name_match_score,query_article_id,query_item_text,rank,score,article_id,item_text,method
0,op t-shirt,boop t-shirt,contains,0.909091,762928001,boop t-shirt t-shirt garment upper body jersey...,1,1.00000,762928002,boop t-shirt t-shirt garment upper body jersey...,tfidf
1,op t-shirt,boop t-shirt,contains,0.909091,762928001,boop t-shirt t-shirt garment upper body jersey...,2,0.82516,681657006,helsinki (1) top garment upper body jersey bas...,tfidf
2,op t-shirt,boop t-shirt,contains,0.909091,762928001,boop t-shirt t-shirt garment upper body jersey...,3,0.82516,681657003,helsinki top garment upper body jersey basic w...,tfidf
3,op t-shirt,boop t-shirt,contains,0.909091,762928001,boop t-shirt t-shirt garment upper body jersey...,4,0.82516,681657005,helsinki (1) top garment upper body jersey bas...,tfidf
4,op t-shirt,boop t-shirt,contains,0.909091,762928001,boop t-shirt t-shirt garment upper body jersey...,5,0.82516,681657001,georgina top garment upper body jersey basic w...,tfidf


Query name: v-neck strap top


,query_name,resolved_prod_name,match_type,name_match_score,query_article_id,query_item_text,rank,score,article_id,item_text,method
0,v-neck strap top,v-neck strap top,exact,1.0,538699001,v-neck strap top vest top garment upper body j...,1,1.000000,538699007,v-neck strap top vest top garment upper body j...,tfidf
1,v-neck strap top,v-neck strap top,exact,1.0,538699001,v-neck strap top vest top garment upper body j...,2,1.000000,538699002,v-neck strap top vest top garment upper body j...,tfidf
2,v-neck strap top,v-neck strap top,exact,1.0,538699001,v-neck strap top vest top garment upper body j...,3,0.877615,108775044,strap top vest top garment upper body jersey b...,tfidf
3,v-neck strap top,v-neck strap top,exact,1.0,538699001,v-neck strap top vest top garment upper body j...,4,0.877615,108775015,strap top vest top garment upper body jersey b...,tfidf
4,v-neck strap top,v-neck strap top,exact,1.0,538699001,v-neck strap top vest top garment upper body j...,5,0.877615,108775051,strap top (1) vest top garment upper body jers...,tfidf
